# Промпт для T-lite: вызов поиска

Продолжение EXP-008 (`tlite_prompt_experiments.ipynb`): правило про `#SLIDE` и пример исправили формат, но поиск
не вызывался ни разу, ни в одном варианте. Вероятная причина — сам промпт: «Ответ — презентация… Пиши только
презентацию, без пояснений до и после» и «Ответ на любой запрос — слайды». Модель так и делает. Здесь проверяю,
вызывается ли поиск при других формулировках. Режим без поиска не тестирую. `prompts.py` не меняю.

Шаг 0 — умеет ли T-lite вообще вызывать инструмент: 4 коротких вопроса о фактах, промпт в одну строку, без формата
презентации.

Варианты (все — с поиском):
- **B** — контроль, как в EXP-008;
- **E** — B, но начало переписано: работа в два этапа, сначала поиск, потом презентация; вызов инструмента описан
  в разделе «Формат» как часть ответа; в описании `#SOURCE` нет образцов id (в EXP-008 модель их копировала);
- **F** — E + пример с поиском: `#PLAN` → вызовы (запросы — пункты плана) → что вернула программа → слайды
  с `#SOURCE`;
- **G** — тот же пример не текстом в промпте, а прошлым диалогом: запрос → реплики с вызовами → настоящие ответы
  Википедии → слайды. Пример собирается той же функцией эпизода (`run_episode`) с заранее написанными репликами
  вместо модели — реплики выглядят ровно так, как выглядели бы её собственные (с пустым `<think>`);
- **H** — F с пояснениями на английском; сами слайды и строки примера — по-русски.

Задания: 8 заданий validation из EXP-008 и 4 запроса с фактами оттуда же — 12 × 5 = 60 эпизодов. T = 0.9, seed
по заданию. Сверх EXP-008 считаю долю кириллицы в тексте слайдов (для H: не перешла ли модель на английский).

Результаты — `runs/search_prompts/<время>/`: шаг 0 — `step0.json`; тексты промптов `prompts.json` и пример G
целиком `g_demo.txt`; по каждому эпизоду `<вариант>_<id>.txt`, `_answer.txt`, `.pptx`; `summary.csv` (после каждого
эпизода) и `totals.csv`.

In [ ]:
import csv, dataclasses, json, os, re, statistics, sys, time
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "4"  # перед запуском проверить nvidia-smi: карта должна быть свободна
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
import torch
from presentation_rl.dsl import parse
from presentation_rl.environment import END, build_prompt, run_episode
from presentation_rl.model_loader import hf_generator, load_model
from presentation_rl.prompts import BASE, SEARCH
from presentation_rl.rendering import build_presentation, pick_theme
from presentation_rl.tasks import PresentationRequest, load_tasks
from presentation_rl.tools.cache import WikiCache
from presentation_rl.tools.wiki import WikiClient
from presentation_rl.tools.wiki_tools import TOOL_SPECS, WikiTools

print(torch.__version__, torch.cuda.get_device_name(0))
CONTACT = os.environ.get("WIKI_CONTACT", "https://github.com/doshik722")
cache = WikiCache(ROOT / "data" / "sources" / "wiki_cache", WikiClient(contact=CONTACT))
OUT = ROOT / "runs" / "search_prompts" / time.strftime("%Y%m%d_%H%M")
OUT.mkdir(parents=True, exist_ok=True)
SEED = 0
print(WikiTools(cache).search("фотоэффект")[:400])  # Википедия с сервера доступна?

In [ ]:
start = time.time()
model, tokenizer = load_model("t-tech/T-lite-it-2.1", dtype="float16")
generate = hf_generator(model, tokenizer, temperature=0.9)
print(f"загрузка {time.time() - start:.0f} с, память {torch.cuda.memory_allocated() / 2**30:.1f} ГБ")

## Шаг 0: вызывает ли T-lite инструмент в простой задаче

Промпт — одна строка, описания инструментов добавляет шаблон чата. Два вопроса прямо просят Википедию, два — нет.
Если вызовов нет и здесь, дело не в нашем промпте, а в формате инструментов.

In [ ]:
QUESTIONS = [
    "Найди в Википедии, в каком году родился Проспер Мериме.",
    "Посмотри в Википедии установленную мощность Саяно-Шушенской ГЭС.",
    "Когда запустили телескоп «Джеймс Уэбб»?",
    "Кто получил Нобелевскую премию по литературе в 1958 году?",
]
step0 = []
for number, question in enumerate(QUESTIONS):
    messages = [{"role": "system", "content": "Ты — полезный помощник с доступом к Википедии."},
                {"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(messages, tools=TOOL_SPECS, add_generation_prompt=True, tokenize=False)
    torch.manual_seed(SEED + number)
    episode = run_episode(tokenizer.encode(text, add_special_tokens=False), tokenizer, generate, WikiTools(cache),
                          max_new_tokens=1000)
    step0.append({"question": question, "calls": episode.calls, "answer": episode.answer, "finish": episode.finish})
    print(question, "\n  вызовы:", [f"{c['name']}({c['argument']})" for c in episode.calls] or "нет",
          "\n  ответ:", episode.answer[:300].replace("\n", " "), "\n")
(OUT / "step0.json").write_text(json.dumps(step0, ensure_ascii=False, indent=1), encoding="utf-8")

## Варианты промпта

B — как в EXP-008: вставки в `BASE` после строк-якорей (если якорь не найден, ячейка падает). E, F, G — замены
в B. H — отдельный текст: перевод F.

In [ ]:
def insert_after(text, anchor, addition):
    assert text.count(anchor) == 1, f"якорь не найден или не один: {anchor}"
    return text.replace(anchor, anchor + addition)


def replace_once(text, old, new):
    assert text.count(old) == 1, f"текст не найден или не один: {old}"
    return text.replace(old, new)


# --- B (как в EXP-008)
SLIDE_RULE = ("\n- Каждый слайд, включая первый (титульный), начинается отдельной строкой `#SLIDE`; строки до первого "
              "`#SLIDE`, кроме `#PLAN`, программа отбрасывает.")
MISTAKES = """

Частые ошибки формата — не допускай их:
- у `#CHART` всегда есть `type=`; в `y` — только числа, по одному на каждую категорию из `x`;
- формулы — только строкой `#FORMULA`; в `#TITLE`, `#TEXT` и `#BULLET` не пиши LaTeX и знаки `\\(`, `\\[`, `$`;
- в строке один тег и его значение; комментарии после значения не пиши."""
EXAMPLE = r"""
## Пример

Начало презентации на другую тему — образец формата, а не содержания и не длины:

#PLAN: открытие → уравнение Эйнштейна → красная граница → применение
#SLIDE
#TITLE: Фотоэффект: как свет выбивает электроны
#TEXT: Почему ультрафиолет разряжает цинковую пластину, а яркий красный свет — нет? Ответ на этот вопрос привёл к идее квантов света.
#IMAGE: металлическая пластина под лучом ультрафиолетовой лампы в физической лаборатории
#SLIDE
#TITLE: Красная граница и выбор металла
#TEXT: Если энергии фотона не хватает на работу выхода, электроны не вылетают при любой яркости. Наименьшая частота — красная граница — у каждого металла своя.
#FORMULA: \nu_{min} = \frac{A}{h}
#CHART: type=bar; title=Работа выхода, эВ; x=цезий,натрий,цинк; y=1.9,2.3,4.2
#TEXT: У цинка граница лежит в ультрафиолете, а у цезия — в видимом свете, поэтому цезий используют в фотоэлементах.
#NOTES: Пояснить, что работа выхода зависит и от состояния поверхности, поэтому в справочниках значения немного различаются. Спросить зал, хватит ли зелёного света, около 2,3 эВ, чтобы выбить электрон из цинка.
"""
BASE_B = insert_after(BASE, "- Перед слайдами — строка `#PLAN`: части презентации по порядку через «→».", SLIDE_RULE)
BASE_B = insert_after(BASE_B, "химию пиши как `6\\text{CO}_2`.", MISTAKES)

# --- E: два этапа, вызов — часть формата, #SOURCE без образцов id
INTRO_E = """Ты делаешь презентации по запросу пользователя. Работа идёт в два этапа. Сначала поиск: ты вызываешь
инструменты и находишь в Википедии факты по теме. Потом презентация в текстовом формате, описанном ниже: из этого
текста программа сама собирает файл PowerPoint. Кроме вызовов инструментов и презентации, ничего не пиши.
"""
CALLS_E = """Поиск. Вызов инструмента — не строка с тегом, а блок:
<tool_call>
{"name": "search", "arguments": {"query": "фотоэффект"}}
</tool_call>
После вызова закончи реплику: программа выполнит вызов и вернёт результат, затем ты продолжишь. В одной реплике
можно сделать несколько вызовов. Перед вызовом — строка `#PLAN`: что и зачем ищешь. Инструменты:
- `search` — `query`: 1–4 ключевых слова → статьи с id (`d1`, `d2`…), языком, названием и началом введения;
- `open` — `id` статьи → введение и оглавление с id разделов (`d1#3`);
- `read` — `id` раздела → текст раздела.

Презентация. """
SEARCH_E = """
## Поиск по Википедии

- Сначала поиск, потом слайды. Ищи определения, даты, числа, имена, данные для графиков — всё, в чём не уверен.
  Обычно хватает 3–6 вызовов, не больше 10.
- Запрос — 1–4 ключевых слова, а не фраза. Кириллица — поиск по русской Википедии, латиница — по английской;
  в технических темах английская часто полнее. Читать можно по-английски, писать — по-русски.
- Найденное видишь только ты, пользователь — нет. Факты из прочитанного переноси точно.
- На слайде с материалом из статьи — строка `#SOURCE` с id раздела или статьи из полученных результатов. Id, которого
  не было в результатах, не пиши: такой источник программа отклонит. Название и ссылку она подставит сама.
"""
BASE_E = replace_once(BASE_B, BASE[:BASE.index("\n\n## Как выполнять запрос") + 1], INTRO_E)
BASE_E = replace_once(BASE_E, "- Ответ на любой запрос — слайды.", "- Итог любого запроса — слайды.")
BASE_E = replace_once(BASE_E, "## Формат\n\nКаждая строка", "## Формат\n\n" + CALLS_E + "Каждая строка")

# --- F: пример с поиском текстом. Реплики те же, что в примере G.
REPLY_1 = """#PLAN: открытие → уравнение Эйнштейна → красная граница → применение
#PLAN: ищу историю открытия, законы фотоэффекта и работу выхода металлов
<tool_call>
{"name": "search", "arguments": {"query": "фотоэффект"}}
</tool_call>"""
REPLY_2 = """<tool_call>
{{"name": "open", "arguments": {{"id": "{article}"}}}}
</tool_call>"""
REPLY_3 = """#PLAN: история открытия — раздел {section}; работа выхода — отдельная статья
<tool_call>
{{"name": "read", "arguments": {{"id": "{section}"}}}}
</tool_call>
<tool_call>
{{"name": "search", "arguments": {{"query": "работа выхода"}}}}
</tool_call>"""
REPLY_4 = r"""#SLIDE
#TITLE: Фотоэффект: как свет выбивает электроны
#TEXT: Почему ультрафиолет разряжает цинковую пластину, а яркий красный свет — нет? Ответ на этот вопрос привёл к идее квантов света.
#IMAGE: металлическая пластина под лучом ультрафиолетовой лампы в физической лаборатории
#SLIDE
#TITLE: От открытия к объяснению
#TEXT: В 1887 году Генрих Герц заметил, что ультрафиолет облегчает проскакивание искры между электродами. Годом позже Александр Столетов начал систематические опыты: освещал металлическую пластину и измерял ток в цепи.
#TIMELINE: 1887 — наблюдение Герца; 1888 — начало опытов Столетова; 1905 — квантовое объяснение Эйнштейна
#SOURCE: {section}
#NOTES: Рассказать, как был устроен опыт Столетова: сетка и сплошная пластина образуют конденсатор, а гальванометр показывает ток только при освещении.
#SLIDE
#TITLE: Красная граница и выбор металла
#TEXT: Если энергии фотона не хватает на работу выхода, электроны не вылетают при любой яркости. Наименьшая частота — красная граница — у каждого металла своя.
#FORMULA: \nu_{{min}} = \frac{{A}}{{h}}
#CHART: type=bar; title=Работа выхода, эВ; x=цезий,натрий,цинк; y=1.9,2.3,4.2
#SOURCE: {work}
#NOTES: Пояснить, что работа выхода зависит и от состояния поверхности, поэтому в справочниках значения немного различаются."""
IDS_F = {"article": "d1", "section": "d1#1", "work": "d3"}
EXAMPLE_F = f"""
## Пример

Запрос «Сделай презентацию о фотоэффекте» — образец формата, а не содержания и не длины. Реплика 1:

{REPLY_1}

Программа возвращает статьи: `d1 | ru | Фотоэффект | Фотоэффект — …`, `d2 | en | Photoelectric effect | …`.
Реплика 2:

{REPLY_2.format(**IDS_F)}

Программа возвращает введение и оглавление: `d1#1 История открытия`, `d1#2 Законы фотоэффекта`… Реплика 3:

{REPLY_3.format(**IDS_F)}

Программа возвращает текст раздела d1#1 и статьи о работе выхода (`d3 | ru | Работа выхода | …`). Фактов
достаточно — реплика 4, презентация (начало):

{REPLY_4.format(**IDS_F)}
"""

# --- H: F с пояснениями на английском
H = r"""You make presentations at the user's request. The work has two stages. First, search: you call tools and find
facts on the topic in Wikipedia. Then, the presentation in the text format described below: a program builds
a PowerPoint file from this text by itself. Write nothing except tool calls and the presentation.

## How to handle the request

- The result of any request is slides. If the user asks for an outline, a script or text for slides, write finished
  slides; if they ask for only a part (introduction, conclusion, one slide), make only that part.
- The number of slides is given in the line «Слайдов» under the request. If there is no such line, make as many as
  needed to cover the topic fully for this audience: usually 10–15; fewer only if the request itself asks for
  something short or only a part. All slides count, including the title slide. Do not add the slides «Спасибо за
  внимание», «Вопросы» or «Источники».
- The audience is the line «Аудитория»; if it is absent, choose the most likely audience for the request. The
  audience knows the basic concepts of its field: cover the topic itself rather than repeating the basics.
- The presentation is in Russian unless the user asks for another language: titles, slide text and notes are in
  Russian; common English terms are acceptable.
- «Тип темы», if present, is a hint: in a technical topic (техническая) mechanisms, formulas and graphs usually
  matter more; in a humanities topic (гуманитарная) — events, causes and consequences, timelines and quotes.
- «Картинки: нет» — do not use `#IMAGE`: use diagrams, graphs, charts, timelines and formulas instead of pictures.
- The text is ready to be shown: no placeholders like «[Ваше имя]». If asked to make something up (a company,
  a project), make it up; do not make up real facts. Write only what you are sure of. Numbers on charts are real;
  mark made-up data in the chart title: «(условный пример)». Quotes are only real ones, with author and source.

## Format

Search. A tool call is not a tagged line but a block:
<tool_call>
{"name": "search", "arguments": {"query": "фотоэффект"}}
</tool_call>
After a call, end your turn: the program runs the call and returns the result, then you continue. One turn may
contain several calls. Before a call — a `#PLAN` line: what you are looking for and why. Tools:
- `search` — `query`: 1–4 keywords → articles with ids (`d1`, `d2`…), language, title and the beginning of the
  introduction;
- `open` — article `id` → introduction and table of contents with section ids (`d1#3`);
- `read` — section `id` → text of the section.

Presentation. Each line is one element of the form `#TAG: value`, the value on one line. Lines without a tag are
not allowed. Do not use Markdown inside values.

- `#PLAN` — plan, not shown on slides: `#PLAN: история → закон → опыт → применение`
- `#SLIDE` — start of a new slide, no value
- `#TITLE` — slide title, exactly one per slide: `#TITLE: Законы фотоэффекта`
- `#TEXT` — a paragraph of connected text: `#TEXT: Свет выбивает электроны из металла, …`
- `#BULLET` — a list item, only for homogeneous items: `#BULLET: Фотоэлементы в датчиках освещённости`
- `#FORMULA` — a formula in LaTeX without $ signs: `#FORMULA: h\nu = A + \frac{m v^2}{2}`
- `#CHART` — a chart over categories; type = bar, line or pie; 2–12 values; decimal point is a dot:
  `#CHART: type=bar; title=Работа выхода, эВ; x=цезий,натрий,цинк; y=1.9,2.3,4.2`
- `#PLOT` — a function graph; allowed: numbers, x, + - * / ^, brackets, sin cos tan exp log sqrt abs, pi, e:
  `#PLOT: y = 1 - exp(-x); x=0..5; title=Насыщение фототока`
- `#DIAGRAM` — a diagram of up to 8 nodes: a flow (arrow label between vertical bars) or a tree:
  `#DIAGRAM: type=flow; Фотон -->|поглощение| Электрон -->|вылет| Ток в цепи`
  `#DIAGRAM: type=tree; Фотоэффект > Внешний, Внутренний; Внутренний > Фоторезистор, Фотодиод`
- `#TIMELINE` — a timeline, 2–8 items «date — event» separated by semicolons:
  `#TIMELINE: 1887 — открытие Герца; 1888 — опыты Столетова; 1905 — объяснение Эйнштейна`
- `#QUOTE` — a quote; after the last « — » the author and source: `#QUOTE: Текст цитаты — Автор, «Произведение»`
- `#IMAGE` — a description of the needed picture, the program will find it: `#IMAGE: вакуумный фотоэлемент со стеклянной колбой`
- `#NOTES` — text for the speaker, in the slide notes, not visible on the slide: `#NOTES: Рассказать, как …`

Format constraints:
- in `#CHART`, `#PLOT` and `#DIAGRAM` the `key=value` pairs are separated by `;`, list items by `,`; the characters
  `;` and `=` are not allowed inside values; the X axis of `#CHART` is categories, a dependence on a number is `#PLOT`;
- labels of `#DIAGRAM` nodes and arrows do not contain `->`, `>`, `|`, `;` or `,`; a diagram has at most 8 nodes;
  in a tree each node has one parent, otherwise use `type=flow`;
- in formulas do not use matrices, `\ce{…}`, `\underbrace`, `\overbrace`; write chemistry as `6\text{CO}_2`.

Common format mistakes — avoid them:
- `#CHART` always has `type=`; `y` contains only numbers, one for each category in `x`;
- formulas only in a `#FORMULA` line; no LaTeX and no `\(`, `\[`, `$` in `#TITLE`, `#TEXT` and `#BULLET`;
- one tag and its value per line; no comments after the value.

## Slides

- Before the slides — a `#PLAN` line: the parts of the presentation in order, separated by «→».
- Every slide, including the first (title) one, starts with a separate `#SLIDE` line; lines before the first
  `#SLIDE`, except `#PLAN`, are discarded by the program.
- The first slide is the title slide: `#TITLE` with the name and one or two `#TEXT` lines on what the presentation
  is about. A slide with only a title is drawn by the program as a section divider.
- A slide usually has 30–60 words of text. Reasoning where a sentence builds on the previous one — `#TEXT`
  paragraphs; homogeneous items — `#BULLET`.
- Every slide has at least one visual element: a diagram, a graph, a timeline, a chart, a formula or a picture,
  chosen by meaning and of different types; the visual does not repeat the text word for word. At most 2 large
  elements (`#CHART`, `#PLOT`, `#DIAGRAM`, `#TIMELINE`, `#IMAGE`) and 3 formulas per slide.
- Pictures are for things that exist as a photo or illustration: people, places, devices, objects, maps. For
  abstract concepts — a diagram, a graph or a formula.
- Diagrams are meaningful: with branches or arrow labels, they show connections that are not in the text.
- Every slide except the title one has `#NOTES` of 2–4 sentences: what the speaker says beyond what is written on
  the slide — explanations, an example, the link to neighbouring slides, a question to the audience.

## Wikipedia search

- Search first, slides after. Look up definitions, dates, numbers, names, data for charts — everything you are not
  sure of. Usually 3–6 calls are enough, at most 10.
- A query is 1–4 keywords, not a sentence. Cyrillic — search in the Russian Wikipedia, Latin — in the English one;
  in technical topics the English one is often more complete. You may read in English, but write in Russian.
- Only you see what was found, the user does not. Transfer facts from what you read exactly.
- On a slide with material from an article — a `#SOURCE` line with the id of a section or an article from the
  results you received. Do not write an id that was not in the results: the program will reject such a source.
  It fills in the title and the link by itself.

## Example

The request «Сделай презентацию о фотоэффекте» — a sample of the format, not of the content or length. Turn 1:

""" + REPLY_1 + """

The program returns articles: `d1 | ru | Фотоэффект | Фотоэффект — …`, `d2 | en | Photoelectric effect | …`.
Turn 2:

""" + REPLY_2.format(**IDS_F) + """

The program returns the introduction and the table of contents: `d1#1 История открытия`, `d1#2 Законы фотоэффекта`…
Turn 3:

""" + REPLY_3.format(**IDS_F) + """

The program returns the text of section d1#1 and the article on the work function (`d3 | ru | Работа выхода | …`).
Enough facts — turn 4, the presentation (beginning):

""" + REPLY_4.format(**IDS_F) + "\n"

### Пример G: прошлый диалог с настоящими ответами Википедии

Реплики «модели» — те же, что в примере F (в запросе примера — 3 слайда), но id подставляются из настоящих результатов (какой раздел — история
открытия, какой id у статьи «Работа выхода»). Ответы инструментов укорочены до 3000 символов, чтобы пример не съел
контекст. Пример печатается целиком — проверить глазами, что ответы Википедии осмысленные.

In [ ]:
def find(pattern, text, fallback):
    match = re.search(pattern, text, re.MULTILINE)
    if match is None:
        print(f"не найдено {pattern!r}, беру {fallback}")
    return match.group(1) if match else fallback


def demo_replies():
    ids = {}

    def reply_2(context):
        ids["article"] = find(r"^(d\d+) \| ru \| Фотоэффект \|", context, "d1")
        return REPLY_2.format(**ids)

    def reply_3(context):
        ids["section"] = find(rf"^({ids['article']}#\d+) +История", context, ids["article"] + "#1")
        return REPLY_3.format(**ids)

    def reply_4(context):
        ids["work"] = find(r"^(d\d+) \| ru \| Работа выхода", context, "d3")
        return REPLY_4.format(**ids)

    return [lambda context: REPLY_1, reply_2, reply_3, reply_4]


def scripted(replies):
    """Вместо модели: по очереди отдаёт заранее написанные реплики (функции от уже собранного текста)."""
    replies = list(replies)

    def generate_reply(ids, max_new_tokens):
        text = replies.pop(0)(tokenizer.decode(ids)) + END
        return tokenizer.encode(text, add_special_tokens=False), True

    return generate_reply


SYSTEM_G = BASE_E + SEARCH_E
# В запросе примера — 3 слайда: иначе короткий пример учил бы делать короткие презентации.
demo_request = PresentationRequest("Сделай презентацию о фотоэффекте.", slides=3, domain="technical")
demo = run_episode(build_prompt(tokenizer, SYSTEM_G, demo_request), tokenizer, scripted(demo_replies()),
                   WikiTools(cache, limit=3000))
assert demo.finish == "stop" and len(demo.calls) == 4, (demo.finish, demo.calls)
G_PREFIX = demo.ids  # системный промпт, описания инструментов и весь пример; заканчивается <|im_end|>
demo_text = tokenizer.decode(G_PREFIX)
(OUT / "g_demo.txt").write_text(demo_text, encoding="utf-8")
print(demo_text[demo_text.index("<|im_start|>user"):])

In [ ]:
TEXTS = {"B": BASE_B + SEARCH + EXAMPLE, "E": BASE_E + SEARCH_E + EXAMPLE, "F": BASE_E + SEARCH_E + EXAMPLE_F, "H": H}


def make_prompt(name, request):
    assert request.search, "здесь только запросы с поиском"
    if name == "G":
        turn = f"\n<|im_start|>user\n{request.render()}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
        return G_PREFIX + tokenizer.encode(turn, add_special_tokens=False)
    return build_prompt(tokenizer, TEXTS[name], request)


VARIANTS = ["B", "E", "F", "G", "H"]
probe = PresentationRequest("Сделай презентацию о фотоэффекте.")
(OUT / "prompts.json").write_text(json.dumps({**TEXTS, "G": SYSTEM_G}, ensure_ascii=False, indent=1), encoding="utf-8")
for name in VARIANTS:
    print(name, len(make_prompt(name, probe)), "токенов промпта вместе с описаниями инструментов и запросом")

In [ ]:
print(TEXTS["F"])  # F целиком; E — то же без примера с поиском, H — перевод F

## Задания и прогон

In [ ]:
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="validation")}
REQUESTS = {key: dataclasses.replace(tasks[key].request, search=True, images=True)
            for key in ["t367", "t369", "t376", "t381", "t366", "t378", "t386", "t389"]}
REQUESTS.update({
    "s1": PresentationRequest("Сделай презентацию о Полтавской битве по материалам Википедии, с указанием источников.",
                              domain="humanities"),
    "s2": PresentationRequest("Презентация о крупнейших ГЭС России: мощность, год ввода в эксплуатацию, река. "
                              "Данные возьми из Википедии.", slides=8, domain="technical"),
    "s3": PresentationRequest("Сделай презентацию о космическом телескопе «Джеймс Уэбб»: когда запущен, какого размера "
                              "зеркало, где находится и что уже открыл.", domain="technical"),
    "s4": PresentationRequest("Презентация о нобелевских лауреатах по литературе из России и СССР: кто, когда и за что "
                              "получил премию.", domain="humanities"),
})

FIELDS = ["variant", "id", "finish", "prompt_tokens", "generated_tokens", "seconds", "slides", "requested", "in_range",
          "starts_with_slide", "parse_rate", "parse_errors", "error_codes", "calls", "call_names", "queries",
          "query_words", "call_errors", "sources_ok", "sources_bad", "cyrillic", "render"]
rows = []


def starts_with_slide(answer):
    for line in answer.splitlines():
        line = line.strip()
        if line and not line.startswith("```") and not line.upper().startswith("#PLAN"):
            return line.upper().startswith("#SLIDE")
    return False


def cyrillic_share(deck):
    # Доля кириллицы среди букв заголовков, текста и пунктов: для H — не перешла ли модель на английский.
    text = " ".join([s.title or "" for s in deck.slides] +
                    [e.data.get("text", "") for s in deck.slides for e in s.elements if e.kind in ("TEXT", "BULLET")])
    letters = [ch for ch in text if ch.isalpha()]
    return round(sum("а" <= ch.lower() <= "я" or ch.lower() == "ё" for ch in letters) / len(letters), 2) if letters else ""


def run(name, key, request, number):
    tools = WikiTools(cache)
    prompt_ids = make_prompt(name, request)
    torch.manual_seed(SEED + number)
    start = time.time()
    episode = run_episode(prompt_ids, tokenizer, generate, tools, max_new_tokens=8000)
    seconds = time.time() - start
    deck = parse(episode.answer)
    stem = f"{name}_{key}"
    report = build_presentation(deck, OUT / f"{stem}.pptx", pick_theme(request.theme, request.text),
                                sources=tools.source, show_sources=request.sources)
    full = tokenizer.decode(episode.ids)
    if name == "G":  # пример G одинаков во всех эпизодах (он в g_demo.txt) — в файл эпизода не пишу
        full = full[len(demo_text):]
    (OUT / f"{stem}.txt").write_text(full, encoding="utf-8")
    (OUT / f"{stem}_answer.txt").write_text(episode.answer, encoding="utf-8")

    requested = request.slide_range()
    queries = [c["argument"] for c in episode.calls if c["name"] == "search" and not c["error"]]
    sources = [item.status for item in report.items if item.kind == "SOURCE"]
    row = {
        "variant": name, "id": key, "finish": episode.finish,
        "prompt_tokens": len(episode.prompt_ids), "generated_tokens": episode.generated_tokens,
        "seconds": round(seconds), "slides": len(deck.slides),
        "requested": "" if requested is None else "–".join(str(n) for n in sorted(set(requested))),
        "in_range": "" if requested is None else requested[0] <= len(deck.slides) <= requested[1],
        "starts_with_slide": starts_with_slide(episode.answer),
        "parse_rate": round(deck.parse_rate, 3), "parse_errors": len(deck.errors),
        "error_codes": json.dumps(deck.error_counts(), ensure_ascii=False),
        "calls": len(episode.calls), "call_names": ",".join(c["name"] or "?" for c in episode.calls),
        "queries": " | ".join(queries),
        "query_words": round(statistics.mean(len(q.split()) for q in queries), 1) if queries else "",
        "call_errors": sum(bool(c["error"]) for c in episode.calls),
        "sources_ok": sources.count("ok"), "sources_bad": sources.count("bad_source"),
        "cyrillic": cyrillic_share(deck),
        "render": json.dumps({k: v for k, v in report.status_counts().items() if k != "ok"}, ensure_ascii=False),
    }
    rows.append(row)
    with open(OUT / "summary.csv", "w", newline="", encoding="utf-8-sig") as f:  # utf-8-sig — для Excel
        writer = csv.DictWriter(f, FIELDS)
        writer.writeheader()
        writer.writerows(rows)
    print(f"{stem}: {row['finish']}, {row['generated_tokens']} ток., {row['seconds']} с, слайдов {row['slides']}"
          f" (просили {row['requested'] or '—'}), ошибок {row['parse_errors']}, вызовов {row['calls']}"
          f" [{row['queries']}], источников {row['sources_ok']}/{row['sources_ok'] + row['sources_bad']},"
          f" кириллица {row['cyrillic']}")
    return row


def save_totals():
    table = []
    for name in VARIANTS:
        group = [r for r in rows if r["variant"] == name]
        if not group:
            continue
        ranged = [r for r in group if r["in_range"] != ""]
        table.append({
            "variant": name, "episodes": len(group),
            "with_calls": sum(r["calls"] > 0 for r in group),
            "calls": sum(r["calls"] for r in group),
            "call_errors": sum(r["call_errors"] for r in group),
            "sources_ok": sum(r["sources_ok"] for r in group),
            "sources_bad": sum(r["sources_bad"] for r in group),
            "no_slides": sum(r["slides"] == 0 for r in group),
            "starts_with_slide": sum(r["starts_with_slide"] for r in group),
            "in_range": f"{sum(r['in_range'] is True for r in ranged)}/{len(ranged)}",
            "slides_mean": round(statistics.mean(r["slides"] for r in group), 1),
            "parse_rate_mean": round(statistics.mean(r["parse_rate"] for r in group), 3),
            "parse_errors": sum(r["parse_errors"] for r in group),
            "cyrillic_min": min((r["cyrillic"] for r in group if r["cyrillic"] != ""), default=""),
            "not_stopped": sum(r["finish"] != "stop" for r in group),
            "tokens_mean": round(statistics.mean(r["generated_tokens"] for r in group)),
            "seconds_mean": round(statistics.mean(r["seconds"] for r in group)),
        })
    with open(OUT / "totals.csv", "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, list(table[0]))
        writer.writeheader()
        writer.writerows(table)
    columns = list(table[0])
    widths = [max(len(c), *(len(str(r[c])) for r in table)) for c in columns]
    print("  ".join(c.ljust(w) for c, w in zip(columns, widths)))
    for r in table:
        print("  ".join(str(r[c]).ljust(w) for c, w in zip(columns, widths)))

Порядок — по заданиям: если прогон прервётся, у готовых заданий будут все пять вариантов.

In [ ]:
for number, (key, request) in enumerate(REQUESTS.items()):
    for name in VARIANTS:
        run(name, key, request, number)
save_totals()

In [ ]:
# Первый эпизод с вызовами целиком — как его видела модель (без системного промпта и примера G).
with_calls = [r for r in rows if r["calls"]]
if with_calls:
    text = (OUT / f"{with_calls[0]['variant']}_{with_calls[0]['id']}.txt").read_text(encoding="utf-8")
    print(text[text.index("<|im_start|>user"):] if "<|im_start|>user" in text else text)
else:
    print("ни одного вызова инструментов")